# Example 1: Translating the Tn3 Transposase Gene with the Standard Genetic Code


Fetches **Transposon Tn3 (E. coli)**, GenBank accession `V00613`, and translates its first
annotated CDS — *tnpA*, the transposase — with the standard genetic code (NCBI translation
table 1), using `Seq.translate(cds=True)` to both translate AND validate that the sequence
is a clean, in-frame coding sequence with no internal stop codons.


**Expected result:** 3,048 bp CDS (positions 34–3081) → 1,015 aa protein, matching the known
TnpA transposase length exactly (≈114.5 kDa). This is the large DDE transposase that drives
Tn3's own replicative transposition — not a lactose-metabolism enzyme.

**Result:** 1,015 residues, matching the known TnpA transposase length exactly (≈114.5 kDa).
`cds=True` completing without error is itself meaningful — it confirms this record's first
CDS is a clean, in-frame, single-stop CDS, worth checking before trusting any translation
downstream.


In [10]:
# Fetch Transposon Tn3 (E. coli), translate tnpA, and validate it
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"

with Entrez.efetch(
    db="nucleotide", id="V00613",
    rettype="gb", retmode="text"
) as handle:
    record = SeqIO.read(handle, "genbank")

SeqIO.write(record, "data/V00613.gb", "genbank")

# Select the intended CDS by protein ID
for feature in record.features:
    if (feature.type == "CDS" and
            "CAA23884.1" in feature.qualifiers.get("protein_id", [])):
        cds_seq = feature.extract(record.seq)
        break
else:
    raise ValueError("Target CDS not found.")

# Translate using the annotated genetic code
protein = cds_seq.translate(table=11, cds=True)

# Validate against the annotated reference protein
reference = feature.qualifiers["translation"][0]
assert str(protein) == reference, "Protein does not match reference!"

lines = [
    f"CDS length: {len(cds_seq)} bp",
    f"Protein length: {len(protein)} aa (stop codon excluded)",
    f"First 30 residues: {protein[:30]}",
    f"Last 10 residues:  {protein[-10:]}",
    "Exact sequence match: True",
]

print("\n".join(lines))

with open("results/example1_tnpA_protein.fasta", "w") as f:
    f.write(f">tnpA_V00613_translated\n{protein}\n")

with open("results/example1_tnpA_summary.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

print("\nValidation passed. Results saved.")

CDS length: 3048 bp
Protein length: 1015 aa (stop codon excluded)
First 30 residues: MLKKPSGREADMPVDFLTTEQVESYGRFTG
Last 10 residues:  KEASEAENVA
Exact sequence match: True

Validation passed. Results saved.
